### The ideia is to from the spikes identified, try to find the source of the peaks accordinf to the signal characteris.

In [1]:
# import the libraries
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib import cm, colors as mcolors



import scienceplots
plt.style.use(['science', 'no-latex'])



In [2]:


def gaussian_score(measured, expected, tolerance=0.30):
    """
    Gaussian similarity score.

    Parameters
    ----------
    measured : float, array, or Series
    expected : float, array, or Series
    tolerance : float
        Fractional tolerance (30% by default)
    """

    measured = np.asarray(measured, dtype=float)
    expected = np.asarray(expected, dtype=float)

    relative_error = np.abs(measured - expected) / expected

    score = np.exp(-(relative_error**2) / (2 * tolerance**2))

    # invalid expected values
    score = np.where(expected <= 0, np.nan, score)

    return score


def identify_signals(
    peaks_df,
    database_df,
    tolerance=0.30,
    frequency_weight=0.5,
    obw_weight=0.5,
):
    """
    Compare detected peaks against a signal database.

    Parameters
    ----------
    peaks_df : DataFrame

        Required columns

        detected_frequency_[MHz]
        occupied_bandwidth_[MHz]
        fwhm_[MHz]
        obw_to_fwhm_ratio

    database_df : DataFrame

        Required columns

        signal
        freq_start_MHz
        freq_stop_MHz
        typical_obw_MHz

    Returns
    -------
    DataFrame

    One row per detected peak.
    """

    results = []

    for _, peak in peaks_df.iterrows():

        freq = peak["fitted_frequency_[MHz]"]
        obw = peak["occupied_bandwidth_[MHz]"]

        # Candidate signals whose allocation contains the frequency
        candidates = database_df[
            (database_df["freq_start_MHz"] <= freq)
            &
            (database_df["freq_stop_MHz"] >= freq)
        ]

        if len(candidates) == 0:

            results.append({

                "frequency_MHz": freq,

                "identified_signal": "Unknown",

                "confidence": 0,

                "obw_score": np.nan,

                "expected_obw_MHz": np.nan,

                "measured_obw_MHz": obw,

                "fwhm_MHz": peak["fwhm_[MHz]"],

                "obw_to_fwhm_ratio": peak["obw_to_fwhm_ratio"]

            })

            continue

        best_score = -1
        best = None

        for _, candidate in candidates.iterrows():

            obw_score = gaussian_score(
                obw,
                candidate["typical_obw_MHz"],
                tolerance=tolerance
            )

            total_score = (
                frequency_weight * 1.0
                +
                obw_weight * obw_score
            )

            if total_score > best_score:

                best_score = total_score

                best = {

                    "frequency_MHz": freq,

                    "identified_signal": candidate["signal"],

                    "confidence": round(total_score,3),

                    "obw_score": round(obw_score,3),

                    "expected_obw_MHz":
                        candidate["typical_obw_MHz"],

                    "measured_obw_MHz": obw,

                    "fwhm_MHz":
                        peak["fwhm_[MHz]"],

                    "obw_to_fwhm_ratio":
                        peak["obw_to_fwhm_ratio"]

                }

        results.append(best)

    return pd.DataFrame(results)

In [3]:
rfi_bands = {

"Cellular GSM Uplink": (1710, 1785),

"Cellular GSM Downlink": (1805, 1880),

"DECT Phones": (1880, 1900),

"UMTS Uplink": (1920, 1980),

"UMTS Downlink": (2110, 2170),

"Mobile Satellite": (2000, 2200),

"Space Research / EESS": (2200, 2290),

"IMT 2300 (4G/5G)": (2300, 2400),

"Wi-Fi / Bluetooth (ISM)": (2400, 2483),

"LTE / 5G (2600 band)": (2500, 2690),

"Air Traffic Radar": (2700, 2900),

"Weather Radar": (2700, 3000),

"Microwave Backhaul": (3000, 3300),

"5G NR (n78)": (3300, 3600)

}


# Load the data

Define parameters here so the notebook can easily be reused with other datasets.

EDD_2023-03-11T14_58_01

EDD_2023-04-18T18_40_20 

EDD_2023-07-14T07_22_49 

EDD_2023-08-07T15_07_54


In [4]:
filename = 'EDD_2023-03-11T14_58_01'
scan = 'scan_000_P1_ND0'
full_path = os.path.join(filename, scan, f'{filename}_high_peak_details.csv')

data = pd.read_csv(full_path)

data

,detected_frequency_[MHz],fitted_frequency_[MHz],fitted_amplitude_[dB],detrended_signal_[dB],signal_amplitude_[dB],occupied_bandwidth_[MHz],fwhm_[MHz],obw_to_fwhm_ratio
0,1819.627762,1819.639282,6.291893,7.746628,254.777336,3.631592,0.562475,6.456455
1,2002.782822,2002.779731,10.225300,11.396878,259.434333,2.029419,0.876954,2.314167
2,2184.148788,2184.147361,0.929195,1.095014,250.041599,0.267029,0.042203,6.327200
3,2199.930191,2199.920505,8.575429,9.563067,259.267510,2.082825,0.873629,2.384107
4,2362.897873,2362.878643,0.996114,1.064483,251.085457,3.631592,1.931197,1.880488
5,2492.006302,2492.012307,4.003498,3.913530,253.888736,0.267029,0.038104,7.007931


# signal characterization

In [5]:

rfi_database = {
    "Cellular GSM Uplink": {
        "freq_start_MHz": 1710, "freq_stop_MHz": 1785, "typical_obw_MHz": 0.20,
        "spectral_shape": "Gaussian", "expected_eta": "1.5 - 2.5", "occupancy": "Dynamic / Burst",
        "notes": "Handset GMSK modulation; power varies by distance to base station."
    },
    "Cellular GSM Downlink": {
        "freq_start_MHz": 1805, "freq_stop_MHz": 1880, "typical_obw_MHz": 0.20,
        "spectral_shape": "Gaussian", "expected_eta": "1.5 - 2.5", "occupancy": "Continuous",
        "notes": "Base station GMSK; constant power broadcast control channels."
    },
    "DECT Phones": {
        "freq_start_MHz": 1880, "freq_stop_MHz": 1900, "typical_obw_MHz": 1.728,
        "spectral_shape": "Gaussian / Rounded", "expected_eta": "2.0", "occupancy": "Transient / Low",
        "notes": "Cordless telephony; short active TDMA/TDD frames with periodic beacons."
    },
    "UMTS Uplink": {
        "freq_start_MHz": 1920, "freq_stop_MHz": 1980, "typical_obw_MHz": 3.84,
        "spectral_shape": "RRC Roll-off", "expected_eta": "2.0 - 2.8", "occupancy": "Dynamic",
        "notes": "3G handset WCDMA; noise-like power spectral density."
    },
    "UMTS Downlink": {
        "freq_start_MHz": 2110, "freq_stop_MHz": 2170, "typical_obw_MHz": 3.84,
        "spectral_shape": "RRC Roll-off", "expected_eta": "2.0 - 2.8", "occupancy": "Continuous",
        "notes": "3G base station; flat noise pedestal with sharp alpha=0.22 roll-off skirts."
    },
    "Mobile Satellite": {
        "freq_start_MHz": 2000, "freq_stop_MHz": 2200, "typical_obw_MHz": 1.0,
        "spectral_shape": "Raised Cosine", "expected_eta": "2.0 - 3.5", "occupancy": "Intermittent",
        "notes": "MSS S-band downlink; low power level, visible Doppler drift."
    },
    "Space Research / EESS": {
        "freq_start_MHz": 2200, "freq_stop_MHz": 2290, "typical_obw_MHz": 2.0,
        "spectral_shape": "Sinc / Central Peak", "expected_eta": "1.5 - 3.0", "occupancy": "Intermittent",
        "notes": "Earth Exploration Satellite; prominent residual carrier tone with phase sidebands."
    },
    "IMT 2300 (4G/5G)": {
        "freq_start_MHz": 2300, "freq_stop_MHz": 2400, "typical_obw_MHz": 10.0,
        "spectral_shape": "Flat-Top (OFDM)", "expected_eta": "> 5.0", "occupancy": "High Load",
        "notes": "LTE B40 / 5G n40 TDD spectrum; steep rectangular filter mask."
    },
    "Wi-Fi / Bluetooth (ISM)": {
        "freq_start_MHz": 2400, "freq_stop_MHz": 2483.5, "typical_obw_MHz": 20.0,
        "spectral_shape": "Flat-Top / Sinc", "expected_eta": "Variable", "occupancy": "Highly Burst",
        "notes": "Unlicensed 2.4 GHz band; packetized Wi-Fi bursts mixed with BT frequency hopping."
    },
    "LTE / 5G (2600 band)": {
        "freq_start_MHz": 2500, "freq_stop_MHz": 2690, "typical_obw_MHz": 20.0,
        "spectral_shape": "Flat-Top (OFDM)", "expected_eta": "> 5.0", "occupancy": "Continuous / Dynamic",
        "notes": "LTE B7/B38/B41 & 5G n41; wideband OFDM carriers."
    },
    "Air Traffic Radar": {
        "freq_start_MHz": 2700, "freq_stop_MHz": 2900, "typical_obw_MHz": 2.0,
        "spectral_shape": "Pulsed Sinc", "expected_eta": "< 1.5", "occupancy": "Periodic Pulsed",
        "notes": "Airport Surveillance Radar; very high peak power, low duty cycle (<1%)."
    },
    "Weather Radar": {
        "freq_start_MHz": 2700, "freq_stop_MHz": 3000, "typical_obw_MHz": 2.0,
        "spectral_shape": "Pulsed Sinc", "expected_eta": "< 1.5", "occupancy": "Periodic Pulsed",
        "notes": "Meteorological Doppler Radars; high peak power during rotational sweeps."
    },
    "Microwave Backhaul": {
        "freq_start_MHz": 3000, "freq_stop_MHz": 3300, "typical_obw_MHz": 14.0,
        "spectral_shape": "Raised Cosine (QAM)", "expected_eta": "2.5 - 4.0", "occupancy": "Continuous",
        "notes": "Point-to-point links; constant power and stable spectral footprint."
    },
    "5G NR (n78)": {
        "freq_start_MHz": 3300, "freq_stop_MHz": 3600, "typical_obw_MHz": 50.0,
        "spectral_shape": "Flat-Top (OFDM)", "expected_eta": "> 6.0", "occupancy": "Dynamic / High",
        "notes": "5G C-band; ultra-wide spectrum blocks with dynamic spatial beamforming."
    }
}

# Convert to DataFrame
df_rfi = pd.DataFrame.from_dict(rfi_database, orient='index')
df_rfi

,freq_start_MHz,freq_stop_MHz,typical_obw_MHz,spectral_shape,expected_eta,occupancy,notes
Cellular GSM Uplink,1710,1785.0,0.200,Gaussian,1.5 - 2.5,Dynamic / Burst,Handset GMSK modulation; power varies by dista...
Cellular GSM Downlink,1805,1880.0,0.200,Gaussian,1.5 - 2.5,Continuous,Base station GMSK; constant power broadcast co...
DECT Phones,1880,1900.0,1.728,Gaussian / Rounded,2.0,Transient / Low,Cordless telephony; short active TDMA/TDD fram...
UMTS Uplink,1920,1980.0,3.840,RRC Roll-off,2.0 - 2.8,Dynamic,3G handset WCDMA; noise-like power spectral de...
UMTS Downlink,2110,2170.0,3.840,RRC Roll-off,2.0 - 2.8,Continuous,3G base station; flat noise pedestal with shar...
Mobile Satellite,2000,2200.0,1.000,Raised Cosine,2.0 - 3.5,Intermittent,"MSS S-band downlink; low power level, visible ..."
Space Research / EESS,2200,2290.0,2.000,Sinc / Central Peak,1.5 - 3.0,Intermittent,Earth Exploration Satellite; prominent residua...
IMT 2300 (4G/5G),2300,2400.0,10.000,Flat-Top (OFDM),> 5.0,High Load,LTE B40 / 5G n40 TDD spectrum; steep rectangul...
Wi-Fi / Bluetooth (ISM),2400,2483.5,20.000,Flat-Top / Sinc,Variable,Highly Burst,Unlicensed 2.4 GHz band; packetized Wi-Fi burs...
LTE / 5G (2600 band),2500,2690.0,20.000,Flat-Top (OFDM),> 5.0,Continuous / Dynamic,LTE B7/B38/B41 & 5G n41; wideband OFDM carriers.


In [15]:

rfi_database_numeric = {
    "Cellular GSM Uplink": {
        "f_start_MHz": 1710.0,
        "f_stop_MHz": 1785.0,
        "obw_nominal_MHz": 0.200,
        "eta_min": 1.5,
        "eta_max": 2.5,
        "eta_nominal": 2.00,
        "fwhm_nominal_MHz": 0.100,
    },
    "Cellular GSM Downlink": {
        "f_start_MHz": 1805.0,
        "f_stop_MHz": 1880.0,
        "obw_nominal_MHz": 0.200,
        "eta_min": 1.5,
        "eta_max": 2.5,
        "eta_nominal": 2.00,
        "fwhm_nominal_MHz": 0.100,
    },
    "DECT Phones": {
        "f_start_MHz": 1880.0,
        "f_stop_MHz": 1900.0,
        "obw_nominal_MHz": 1.728,
        "eta_min": 1.8,
        "eta_max": 2.2,
        "eta_nominal": 2.00,
        "fwhm_nominal_MHz": 0.864,
    },
    "UMTS Uplink": {
        "f_start_MHz": 1920.0,
        "f_stop_MHz": 1980.0,
        "obw_nominal_MHz": 3.840,
        "eta_min": 2.0,
        "eta_max": 2.8,
        "eta_nominal": 2.40,
        "fwhm_nominal_MHz": 1.600,
    },
    "UMTS Downlink": {
        "f_start_MHz": 2110.0,
        "f_stop_MHz": 2170.0,
        "obw_nominal_MHz": 3.840,
        "eta_min": 2.0,
        "eta_max": 2.8,
        "eta_nominal": 2.40,
        "fwhm_nominal_MHz": 1.600,
    },
    "Mobile Satellite": {
        "f_start_MHz": 2000.0,
        "f_stop_MHz": 2200.0,
        "obw_nominal_MHz": 1.000,
        "eta_min": 2.0,
        "eta_max": 3.5,
        "eta_nominal": 2.75,
        "fwhm_nominal_MHz": 0.364,
    },
    "Space Research / EESS": {
        "f_start_MHz": 2200.0,
        "f_stop_MHz": 2290.0,
        "obw_nominal_MHz": 2.000,
        "eta_min": 1.5,
        "eta_max": 3.0,
        "eta_nominal": 2.25,
        "fwhm_nominal_MHz": 0.889,
    },
    "IMT 2300 (4G/5G)": {
        "f_start_MHz": 2300.0,
        "f_stop_MHz": 2400.0,
        "obw_nominal_MHz": 10.000,
        "eta_min": 5.0,
        "eta_max": 10.0,
        "eta_nominal": 6.50,
        "fwhm_nominal_MHz": 1.538,
    },
    "Wi-Fi / Bluetooth (ISM)": {
        "f_start_MHz": 2400.0,
        "f_stop_MHz": 2483.5,
        "obw_nominal_MHz": 20.000,
        "eta_min": 1.0,
        "eta_max": 8.0,
        "eta_nominal": 4.50,
        "fwhm_nominal_MHz": 4.444,
    },
    "LTE / 5G (2600 band)": {
        "f_start_MHz": 2500.0,
        "f_stop_MHz": 2690.0,
        "obw_nominal_MHz": 20.000,
        "eta_min": 5.0,
        "eta_max": 10.0,
        "eta_nominal": 6.50,
        "fwhm_nominal_MHz": 3.077,
    },
    "Air Traffic Radar": {
        "f_start_MHz": 2700.0,
        "f_stop_MHz": 2900.0,
        "obw_nominal_MHz": 2.000,
        "eta_min": 0.5,
        "eta_max": 1.5,
        "eta_nominal": 1.00,
        "fwhm_nominal_MHz": 2.000,
    },
    "Weather Radar": {
        "f_start_MHz": 2700.0,
        "f_stop_MHz": 3000.0,
        "obw_nominal_MHz": 2.000,
        "eta_min": 0.5,
        "eta_max": 1.5,
        "eta_nominal": 1.00,
        "fwhm_nominal_MHz": 2.000,
    },
    "Microwave Backhaul": {
        "f_start_MHz": 3000.0,
        "f_stop_MHz": 3300.0,
        "obw_nominal_MHz": 14.000,
        "eta_min": 2.5,
        "eta_max": 4.0,
        "eta_nominal": 3.25,
        "fwhm_nominal_MHz": 4.308,
    },
    "5G NR (n78)": {
        "f_start_MHz": 3300.0,
        "f_stop_MHz": 3600.0,
        "obw_nominal_MHz": 50.000,
        "eta_min": 6.0,
        "eta_max": 12.0,
        "eta_nominal": 7.50,
        "fwhm_nominal_MHz": 6.667,
    },
}

df_rfi_num = pd.DataFrame.from_dict(rfi_database_numeric, orient="index")
df_rfi_with_col = df_rfi_num.reset_index(names="System")
df_rfi_with_col

,System,f_start_MHz,f_stop_MHz,obw_nominal_MHz,eta_min,eta_max,eta_nominal,fwhm_nominal_MHz
0,Cellular GSM Uplink,1710.0,1785.0,0.200,1.5,2.5,2.00,0.100
1,Cellular GSM Downlink,1805.0,1880.0,0.200,1.5,2.5,2.00,0.100
2,DECT Phones,1880.0,1900.0,1.728,1.8,2.2,2.00,0.864
3,UMTS Uplink,1920.0,1980.0,3.840,2.0,2.8,2.40,1.600
4,UMTS Downlink,2110.0,2170.0,3.840,2.0,2.8,2.40,1.600
5,Mobile Satellite,2000.0,2200.0,1.000,2.0,3.5,2.75,0.364
6,Space Research / EESS,2200.0,2290.0,2.000,1.5,3.0,2.25,0.889
7,IMT 2300 (4G/5G),2300.0,2400.0,10.000,5.0,10.0,6.50,1.538
8,Wi-Fi / Bluetooth (ISM),2400.0,2483.5,20.000,1.0,8.0,4.50,4.444
9,LTE / 5G (2600 band),2500.0,2690.0,20.000,5.0,10.0,6.50,3.077


In [12]:
def gaussian_similarity(measured, expected, tolerance=0.30):
    """
    Similarity between 0 and 1.

    1 -> perfect agreement
    0 -> very poor agreement
    """

    if pd.isna(expected):
        return np.nan

    if expected <= 0:
        return np.nan

    error = abs(measured - expected) / expected

    return np.exp(-(error**2)/(2*tolerance**2))

In [24]:
def evaluate_detected_peaks(
    peaks_df,
    signal_database,
    tolerance=0.30
):
    """
    Evaluate every detected peak against the radio-service database.

    Parameters
    ----------
    peaks_df : pandas.DataFrame

    signal_database : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
    """

    results = []

    for _, peak in peaks_df.iterrows():

        frequency = peak["detected_frequency_[MHz]"]
        obw = peak["occupied_bandwidth_[MHz]"]
        fwhm = peak["fwhm_[MHz]"]
        

        # ---------------------------------------------------------
        # Find candidate band
        # ---------------------------------------------------------

        candidates = signal_database[
            (signal_database["f_start_MHz"] <= frequency)
            &
            (signal_database["f_stop_MHz"] >= frequency)
        ]

        if len(candidates) == 0:

            results.append({
                
                "frequency_MHz": frequency,
                "allocated_band": "Unknown",
                "band_match": False,
                "obw_score": np.nan,
                "fwhm_score": np.nan,
                "overall_score": 0,
                "conclusion": "Frequency not inside any known allocation."

            })

            continue

        for _, band in candidates.iterrows():

            obw_score = gaussian_similarity(
                obw,
                band["obw_nominal_MHz"],
                tolerance
            )

            fwhm_score = gaussian_similarity(
                fwhm,
                band["fwhm_nominal_MHz"],
                tolerance
            )

            overall = np.nanmean([
                obw_score,
                fwhm_score
            ])

            if overall >= 0.80:

                conclusion = (
                    "Measured characteristics are strongly "
                    "consistent with the allocated service."
                )

            elif overall >= 0.50:

                conclusion = (
                    "Frequency belongs to this allocation, "
                    "but only part of the spectral "
                    "characteristics agree."
                )

            else:

                conclusion = (
                    "Frequency is inside the allocation, "
                    "however the measured spectral "
                    "characteristics differ significantly "
                    "from the expected system."
                )

            results.append({

                "frequency_MHz":
                    frequency,

                "allocated_band":
                    band,

                "band_match":
                    True,

                "expected_obw_MHz":
                    band["fwhm_nominal_MHz"],

                "measured_obw_MHz":
                    obw,

                "obw_score":
                    round(obw_score,3),

                "expected_fwhm_MHz":
                    band["fwhm_nominal_MHz"],

                "measured_fwhm_MHz":
                    fwhm,

                "fwhm_score":
                    round(fwhm_score,3),

                "overall_score":
                    round(overall,3),

                "conclusion":
                    conclusion

            })

    return pd.DataFrame(results)

In [25]:
rfi_evaluation = evaluate_detected_peaks(data, df_rfi_with_col, tolerance=0.30)
rfi_evaluation

,frequency_MHz,allocated_band,band_match,expected_obw_MHz,measured_obw_MHz,obw_score,expected_fwhm_MHz,measured_fwhm_MHz,fwhm_score,overall_score,conclusion
0,1819.627762,System Cellular GSM Downlink f_st...,True,0.100,3.631592,0.000,0.100,0.562475,0.000,0.000,"Frequency is inside the allocation, however th..."
1,2002.782822,System Mobile Satellite f_start_M...,True,0.364,2.029419,0.003,0.364,0.876954,0.000,0.001,"Frequency is inside the allocation, however th..."
2,2184.148788,System Mobile Satellite f_start_M...,True,0.364,0.267029,0.051,0.364,0.042203,0.013,0.032,"Frequency is inside the allocation, however th..."
3,2199.930191,System Mobile Satellite f_start_M...,True,0.364,2.082825,0.001,0.364,0.873629,0.000,0.001,"Frequency is inside the allocation, however th..."
4,2362.897873,System IMT 2300 (4G/5G) f_start_M...,True,1.538,3.631592,0.105,1.538,1.931197,0.696,0.400,"Frequency is inside the allocation, however th..."
5,2492.006302,Unknown,False,NaN,NaN,NaN,NaN,NaN,NaN,0.000,Frequency not inside any known allocation.


In [27]:
rfi_evaluation['conclusion'].values

<StringArray>
['Frequency is inside the allocation, however the measured spectral characteristics differ significantly from the expected system.',
 'Frequency is inside the allocation, however the measured spectral characteristics differ significantly from the expected system.',
 'Frequency is inside the allocation, however the measured spectral characteristics differ significantly from the expected system.',
 'Frequency is inside the allocation, however the measured spectral characteristics differ significantly from the expected system.',
 'Frequency is inside the allocation, however the measured spectral characteristics differ significantly from the expected system.',
                                                                                       'Frequency not inside any known allocation.']
Length: 6, dtype: str

In [ ]:
for i in frequency_measured:
    print(i)

1819.6392818354848
2002.779730550325
2184.147361221589
2199.92050519698
2362.878643137037
2492.012307131787
